## Scraping lyrics from Genius

Genius.com has a web API service for retrieving song URLs by artist's name. 
You can start by setting up a token to let you access their API and pull the song URLs. 
* Here's [documentation for the Genius API](https://docs.genius.com/#/getting-started-h1).
* And here's [info on the access token you need](https://docs.genius.com/#authentication).

But usually we want to retrieve lyrics based on albums (e.g. compare early vs. late albums by an artist). This is trickier to retrieve: we've never had much luck with this directly from Genius, but it helps to work with a combination of Genius and BeautifulSoup. Also, when we start pulling songs with Genius's API we've typically pulled in ads and had to try to clean them out. We want to scrape more cleanly and dodge the ads, so that is what we're attempting in this notebook.


### Working with API info: never push your API details to GitHub!

When you set up your API client, you're going to need [to create (or log in with) a Genius.com account](https://genius.com/signup), and once you're logged in it will send you here to [the API Clients page](https://genius.com/api-clients). 
To set up an API token Genius is going to require that you give them an "App URL": just give them one of your websites, and where it asks for a logo, give them a link to an image on your site. (It's looking for a favicon, and I have one stored at the root of my newtfire website at favicon.ico. So that is what I used.)  You'll receive a client secret and an access token. Copy and store them somewhere secret and safe.

Your Python script is going to need to access these things, but you must NOT push your secret API access info to GitHub! 

**So, here's what we do to secure your Python scraping script:**
* You create a system "dot" file that must be named `.env`
* Save it with your Python script or somewhere in your repo. (It's easiest for you if it's with your Python script.)
* Edit your `.gitignore` file (or set this up if you don't have it!): Add `.env` to it (or make sure it is already there). Storing that string in your `.gitignore` prevents it from being pushed to GitHub.
    * Double check that `.env` is properly screened out after you save it and go to `git add` your directory with the Python and `.env` file.
      NOTE: You should NOT see the `.env` being staged by git and should NEVER be in a git commit. We discovered that our textAnalysis-Hub .gitignore had been screening `.env/` as a directory and not the file, so we needed to add a line to our `.gitignore` while writing this notebook! 
* Your Python script will need to read the secret system dot `.env` file, and there's actually a handy Python library for that called `python-dotenv`.
     * Navigate to your directory that holds your .venv and activate it
     * `pip install python-dotenv`



In [ ]:
import requests
import time
import os
import re  # this is for regex substitutions! 
from dotenv import load_dotenv, find_dotenv
from bs4 import BeautifulSoup

load_dotenv(find_dotenv())

GENIUS_TOKEN = os.getenv("GENIUS_ACCESS_TOKEN")
headers = {"Authorization": f"Bearer {GENIUS_TOKEN}"}
# ebb: CAREFUL! Don't print this to output in your notebook and then push the notebook with all its return cells! 
# Always clear your notebook console by going to Kernel >> Restart Notebook and Clear Outputs of All Cells

### Scraping an album's lyrics

In [ ]:
## ebb: THIS CELL IS FOR TESTING TO SEE WHAT <div> ELEMENTS ARE INSIDE THE data-lyrics-container
## SKIP if you don't need it. :-) 
r = requests.get("https://genius.com/Paramore-whoa-lyrics")
soup = BeautifulSoup(r.text, "html.parser")

lyric_divs = soup.find_all("div", attrs={"data-lyrics-container": "true"})
for i, div in enumerate(lyric_divs):
    print(f"--- div {i} ---")
    print(div.get_text()[:200])
    print()

In [ ]:
# SKIP (ALSO diagnostic for debugging data on albums)

r = requests.get("https://genius.com/albums/Paramore/All-we-know-is-falling")
soup = BeautifulSoup(r.text, "html.parser")

song_links = soup.find_all("a", href=lambda v: v and v.endswith("-lyrics"))
for link in song_links:
    # Print the parent's class so we can see what container each link lives in
    print(repr(link.get_text(strip=True)), "→", link.parent.get("class"))






In [ ]:
# ebb: Here are two scraping functions that work as of 30 March 2026! 
# READ FROM THE BOTTOM TO TOP. 
# Supply your album title and define your relative path to your output directory where you want to store lyrics text files (one text file for each song).
# That information is needed to start up the get_lyrics_from_album_url(album_url, output_dir)
# And as that function pulls song titles from the album page, it then triggers the get_lyrics() function, sending it the song URL.
# get_lyrics(), the first function you see here, delivers the song lyrics back to the main function, which then writes the lyrics to output.
# Genius keeps changing its HTML page metadata, so we probably will have to keep updating this cell over time.
def get_lyrics(song_url):
    r = requests.get(song_url)
    soup = BeautifulSoup(r.text, "html.parser")
    
    lyric_divs = soup.find_all("div", attrs={"data-lyrics-container": "true"})
    
    lyrics = []
    for div in lyric_divs:
        for br in div.find_all("br"):
            br.replace_with("\n")
        text = div.get_text()
        
        if not text.strip():
            continue
        
        # If the div contains a bracket marker, trim everything before the first one
        first_bracket = text.find("[")
        if first_bracket != -1:
            text = text[first_bracket:]
            
        # ebb: The next two lines remove the square-bracketed stuff 
        # like `[Verse 1]`, `[Outro]`, etc. with the regular expression`\[.*?\]` 
        # If you want to KEEP these for use in tagging later for an XML project, 
        # comment out the next two lines. 
        text = re.sub(r'\[.*?\]', '', text)
        text = re.sub(r'\n{3,}', '\n\n', text).strip()
        lyrics.append(text)
    
    return "\n".join(lyrics).strip()
    
def get_lyrics_from_album_url(album_url, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    r = requests.get(album_url)
    soup = BeautifulSoup(r.text, "html.parser")
    
    # Only look for lyrics links inside chart_row-content containers
    track_rows = soup.find_all("div", class_="chart_row-content")
    
    corpus = {}
    for row in track_rows:
        link = row.find("a", href=lambda v: v and v.endswith("-lyrics"))
        if link:
            song_url = link["href"]
            song_title = link.get_text(strip=True)
            print(f"Fetching: {song_title}")
            lyrics = get_lyrics(song_url)
            # This runs the function above!!!
            corpus[song_title] = lyrics
            
            # Clean up (remove spaces, path separators, etc. from the song title) for use as a filename
            # Also remove the "Lyrics" from the end of the song title with a regex subsitutitons on Lyrics$ (at the end of the string), 
            # and removing punctuation from titles with a regex character class.
            safe_title = song_title.replace("/", "-").replace(" ", "_").strip()
            safe_title = re.sub(r'Lyrics$', '', safe_title).strip()
            safe_title = re.sub(r"[,.'!?;:]", '', safe_title)
            filepath = os.path.join(output_dir, f"{safe_title}.txt")
            
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(lyrics)
                
            print(f"  Saved: {filepath}")
            time.sleep(1)
    
    return corpus

# Usage
album_url = "https://genius.com/albums/Paramore/All-we-know-is-falling"
output_dir = "output/paramore-all-we-know-is-falling"
# album_url = "https://genius.com/albums/Paramore/Riot"
# output_dir="output/riot"
corpus = get_lyrics_from_album_url(album_url, output_dir)